# SentimentScope (Part 2): Fine-Tuning DistilBERT for 90%+ Accuracy

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/franz-ob/ML-SentimentScope-AWS-project-2/blob/main/SentimentScope_Finetune.ipynb)

The baseline notebook trained a small transformer **from scratch** and reached 75.51% test accuracy. A model that starts from random weights has to learn English from only 22,500 reviews, which limits how far it can go.

This notebook keeps the same task and data, but starts from a **pretrained** `distilbert-base-uncased` and fine-tunes it. It also:

- saves the best checkpoint to disk,
- writes an `inference.py` module (`SentimentClassifier`) that loads the checkpoint and predicts on batches of reviews.

**How to run:** open in Google Colab, choose *Runtime → Change runtime type → T4 GPU*, then *Runtime → Run all*.

## 1. Setup

In [ ]:
!pip install -q transformers scikit-learn

In [ ]:
import os, re, json, random, time, tarfile, urllib.request

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import classification_report, confusion_matrix

# ---- Hyperparameters (the changes vs. the from-scratch baseline) ----
SEED = 42
MODEL_NAME = "distilbert-base-uncased"  # pretrained encoder (baseline: random init)
MAX_LENGTH = 256                        # baseline: 128
BATCH_SIZE = 32
EPOCHS = 3
LR = 2e-5                               # small LR: we are fine-tuning, not training from scratch
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
CKPT_DIR = "checkpoints/distilbert-imdb"

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"  # mixed precision: faster and lighter on GPU
print("Using device:", device)
if device.type == "cpu":
    print("WARNING: no GPU found. Training on CPU will take hours. "
          "In Colab: Runtime > Change runtime type > T4 GPU.")

## 2. Load the data

In [ ]:
URL = "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz"
if not os.path.isdir("aclImdb"):
    print("Downloading IMDB (about 80 MB)...")
    urllib.request.urlretrieve(URL, "aclImdb_v1.tar.gz")
    with tarfile.open("aclImdb_v1.tar.gz") as tar:
        tar.extractall(".")
print("Dataset ready.")

In [ ]:
def clean_text(text):
    """IMDB reviews contain HTML line breaks; replace them with spaces."""
    return re.sub(r"<br\s*/?>", " ", str(text)).strip()

def load_folder(folder):
    reviews = []
    for filename in sorted(os.listdir(folder)):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                reviews.append(clean_text(f.read()))
    return reviews

train_pos = load_folder("aclImdb/train/pos")
train_neg = load_folder("aclImdb/train/neg")
test_pos = load_folder("aclImdb/test/pos")
test_neg = load_folder("aclImdb/test/neg")

train_df = pd.DataFrame({"review": train_pos + train_neg,
                         "label": [1] * len(train_pos) + [0] * len(train_neg)})
test_df = pd.DataFrame({"review": test_pos + test_neg,
                        "label": [1] * len(test_pos) + [0] * len(test_neg)})

assert len(train_df) == 25000 and len(test_df) == 25000

# Same 90/10 shuffled split as the baseline notebook
shuffled = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
train_size = int(0.9 * len(shuffled))
train_data, val_data = shuffled.iloc[:train_size], shuffled.iloc[train_size:]
print(f"train: {len(train_data)}  val: {len(val_data)}  test: {len(test_df)}")

## 3. Tokenize and build DataLoaders

Reviews are tokenized once up front. Batches are padded to the longest sequence in the batch (dynamic padding) instead of always padding to the maximum length.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class IMDBDataset(Dataset):
    def __init__(self, df, tokenizer, max_length):
        self.encodings = tokenizer(df["review"].tolist(), truncation=True, max_length=max_length)
        self.labels = df["label"].tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: values[idx] for key, values in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_ds = IMDBDataset(train_data, tokenizer, MAX_LENGTH)
val_ds = IMDBDataset(val_data, tokenizer, MAX_LENGTH)
test_ds = IMDBDataset(test_df, tokenizer, MAX_LENGTH)

collator = DataCollatorWithPadding(tokenizer)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE * 2, shuffle=False, collate_fn=collator)
print("Batches per epoch:", len(train_loader))

## 4. Load the pretrained model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label={0: "negative", 1: "positive"},
    label2id={"negative": 0, "positive": 1},
).to(device)
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 5. Train

In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    """Return (accuracy %, predictions, labels)."""
    model.eval()
    all_preds, all_labels = [], []
    for batch in loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        labels = batch.pop("labels")
        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(**batch).logits
        all_preds.append(logits.argmax(dim=-1).cpu())
        all_labels.append(labels.cpu())
    preds = torch.cat(all_preds).numpy()
    labels = torch.cat(all_labels).numpy()
    return float((preds == labels).mean() * 100), preds, labels

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
total_steps = EPOCHS * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

best_val_acc = 0.0
history = []

for epoch in range(EPOCHS):
    model.train()
    running_loss, start = 0.0, time.time()

    for step, batch in enumerate(train_loader):
        batch = {k: v.to(device) for k, v in batch.items()}
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=use_amp):
            loss = model(**batch).loss          # labels in the batch -> HF computes cross-entropy
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        running_loss += loss.item()
        if (step + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{step+1}/{len(train_loader)}] "
                  f"Loss: {running_loss / 100:.4f}")
            running_loss = 0.0

    val_acc, _, _ = evaluate(model, val_loader)
    history.append(val_acc)
    print(f"Epoch {epoch+1} - Validation Accuracy: {val_acc:.2f}%  ({time.time() - start:.0f}s)")

    # Save a checkpoint whenever validation accuracy improves
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        os.makedirs(CKPT_DIR, exist_ok=True)
        model.save_pretrained(CKPT_DIR)
        tokenizer.save_pretrained(CKPT_DIR)
        print(f"  -> saved best checkpoint to {CKPT_DIR}")

print(f"\nBest validation accuracy: {best_val_acc:.2f}%")

## 6. Test the best checkpoint

The test set is only used here, once, after training is finished.

In [ ]:
best_model = AutoModelForSequenceClassification.from_pretrained(CKPT_DIR).to(device)
test_acc, preds, labels = evaluate(best_model, test_loader)
print(f"Test Accuracy: {test_acc:.2f}%\n")
print(classification_report(labels, preds, target_names=["negative", "positive"], digits=4))

cm = confusion_matrix(labels, preds)
fig, ax = plt.subplots(figsize=(4.5, 4))
ax.imshow(cm, cmap="Blues")
ax.set_xticks([0, 1]); ax.set_xticklabels(["negative", "positive"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["negative", "positive"])
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title("Test Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black")
plt.tight_layout()
os.makedirs("docs/figures", exist_ok=True)
plt.savefig("docs/figures/confusion_matrix.png", dpi=150)
plt.show()

report = classification_report(labels, preds, target_names=["negative", "positive"], output_dict=True)
results = {
    "model": MODEL_NAME, "max_length": MAX_LENGTH, "epochs": EPOCHS, "lr": LR,
    "batch_size": BATCH_SIZE, "best_val_accuracy": round(best_val_acc, 2),
    "test_accuracy": round(test_acc, 2),
    "precision": round(report["macro avg"]["precision"], 4),
    "recall": round(report["macro avg"]["recall"], 4),
    "f1": round(report["macro avg"]["f1-score"], 4),
}
with open("results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\nRow for the README results table:")
print(f"| Fine-tuned DistilBERT | {best_val_acc:.2f}% | {test_acc:.2f}% |")
print(f"Macro precision / recall / F1: {results['precision']} / {results['recall']} / {results['f1']}")
if test_acc < 90:
    print("\nBelow 90%: try MAX_LENGTH = 512 with BATCH_SIZE = 16, or MODEL_NAME = 'roberta-base'.")

## 7. Inference interface

The next cell writes `inference.py`, which contains a `SentimentClassifier` class. It loads the saved checkpoint once and classifies a batch of reviews in one call. The same file is in the GitHub repo.

In [ ]:
%%writefile inference.py
"""
SentimentScope inference interface.

Loads a fine-tuned checkpoint (a local folder or a Hugging Face Hub id) and
predicts the sentiment of a batch of movie reviews.

Python:
    from inference import SentimentClassifier

    clf = SentimentClassifier("checkpoints/distilbert-imdb")
    clf.predict(["Loved every minute of it!", "A total waste of time."])

Command line:
    python inference.py --model_path checkpoints/distilbert-imdb "Loved it!" "Awful."
"""

import argparse
import re
from typing import Dict, List, Optional, Sequence, Union

import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

DEFAULT_MAX_LENGTH = 256  # must match the MAX_LENGTH used during training
DEFAULT_LABELS = {0: "negative", 1: "positive"}


def clean_text(text: str) -> str:
    """Replace the HTML line breaks that appear in IMDB reviews."""
    return re.sub(r"<br\s*/?>", " ", str(text)).strip()


class SentimentClassifier:
    """Load a saved checkpoint once, then classify any number of reviews."""

    def __init__(
        self,
        model_path: str,
        max_length: int = DEFAULT_MAX_LENGTH,
        batch_size: int = 64,
        device: Optional[str] = None,
    ):
        self.max_length = max_length
        self.batch_size = batch_size
        self.device = torch.device(
            device or ("cuda" if torch.cuda.is_available() else "cpu")
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_path)
        self.model.to(self.device).eval()

        id2label = {int(k): v for k, v in (self.model.config.id2label or {}).items()}
        if not id2label or all(str(v).startswith("LABEL_") for v in id2label.values()):
            id2label = DEFAULT_LABELS
        self.id2label = id2label

    @torch.no_grad()
    def predict_proba(self, texts: Union[str, Sequence[str]]) -> torch.Tensor:
        """Return class probabilities with shape (n_texts, 2): [negative, positive]."""
        if isinstance(texts, str):
            texts = [texts]
        texts = [clean_text(t) for t in texts]

        all_probs = []
        for start in range(0, len(texts), self.batch_size):
            chunk = texts[start : start + self.batch_size]
            encoded = self.tokenizer(
                chunk,
                truncation=True,
                max_length=self.max_length,
                padding=True,
                return_tensors="pt",
            ).to(self.device)
            logits = self.model(**encoded).logits
            all_probs.append(torch.softmax(logits.float(), dim=-1).cpu())

        if not all_probs:
            return torch.empty(0, self.model.config.num_labels)
        return torch.cat(all_probs)

    def predict(self, texts: Union[str, Sequence[str]]) -> List[Dict[str, Union[str, float]]]:
        """Classify a batch of reviews.

        Returns one dict per review:
            {"label": "positive", "confidence": 0.9731, "positive_probability": 0.9731}
        """
        results = []
        for probs in self.predict_proba(texts):
            idx = int(probs.argmax())
            results.append(
                {
                    "label": self.id2label.get(idx, str(idx)),
                    "confidence": round(float(probs[idx]), 4),
                    "positive_probability": round(float(probs[1]), 4),
                }
            )
        return results


def main() -> None:
    parser = argparse.ArgumentParser(description="Classify movie review sentiment.")
    parser.add_argument("--model_path", required=True, help="Checkpoint folder or Hub id")
    parser.add_argument("--max_length", type=int, default=DEFAULT_MAX_LENGTH)
    parser.add_argument("texts", nargs="+", help="One or more reviews (quote each one)")
    args = parser.parse_args()

    clf = SentimentClassifier(args.model_path, max_length=args.max_length)
    for text, result in zip(args.texts, clf.predict(args.texts)):
        print(f"{result['label']:<9} {result['confidence']:.1%}  {text[:80]}")


if __name__ == "__main__":
    main()

Use it exactly as someone else would after cloning the repo:

In [ ]:
from inference import SentimentClassifier

clf = SentimentClassifier(CKPT_DIR, max_length=MAX_LENGTH)

samples = [
    "The movie was a rollercoaster of emotions, and I loved every moment of it!",
    "The plot was predictable, and the acting was subpar. A waste of time.",
    "Not bad, but I expected more from the director.",
]
for text, result in zip(samples, clf.predict(samples)):
    print(result, "<-", text[:60])

Sanity check: the interface should give almost the same accuracy as the evaluation loop above.

In [ ]:
subset = test_df.sample(1000, random_state=SEED)
out = clf.predict(subset["review"].tolist())
pred_labels = [1 if r["label"] == "positive" else 0 for r in out]
print(f"Accuracy via SentimentClassifier on 1,000 test reviews: "
      f"{np.mean(np.array(pred_labels) == subset['label'].values) * 100:.2f}%")

## 8. Download the results

Run this in Colab to download the checkpoint (zipped), `inference.py`, `results.json` and the confusion matrix. The checkpoint is about 250 MB, which is too large for a normal GitHub upload (see the README for hosting options).

In [ ]:
import shutil
shutil.make_archive("distilbert-imdb-checkpoint", "zip", CKPT_DIR)

try:
    from google.colab import files
    for path in ["distilbert-imdb-checkpoint.zip", "inference.py", "results.json",
                 "docs/figures/confusion_matrix.png"]:
        files.download(path)
except ImportError:
    print("Not running in Colab. Files are in the current folder:",
          "distilbert-imdb-checkpoint.zip, inference.py, results.json, docs/figures/confusion_matrix.png")

## Conclusion

*After running, fill this in with your real numbers from the cells above.*

- **Baseline (from scratch):** 75.51% test accuracy.
- **Fine-tuned DistilBERT:** `__` % test accuracy (see Section 6).
- **Why it improved:** the pretrained model already understands language, so the small labeled dataset only has to teach it the sentiment task. The longer context (256 vs 128 tokens) and a proper learning-rate schedule (warmup plus linear decay) also help.
- **Deliverables:** a saved checkpoint and `SentimentClassifier` in `inference.py` for batch inference.